In [2]:
import socket
import json
import time
from pynq import Overlay

# -----------------------------
# Load overlay
# -----------------------------
ol = Overlay("/home/xilinx/jupyter_notebooks/mlp/mlp.bit")
mlp = ol.mlp_axi_v1_0_MLP_AXI_0

CLASS_NAMES = ["circle", "rectangle", "triangle", "line", "freehand"]

# -----------------------------
# UDP setup
# -----------------------------
UDP_IP = "0.0.0.0"
UDP_PORT = 5005

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))

print(f"[udp] listening on {UDP_IP}:{UDP_PORT}")

# default reply destination if packet doesn't include one
DEFAULT_REPLY_IP = "192.168.2.1"
DEFAULT_REPLY_PORT = 5006

# -----------------------------
# AXI helpers
# -----------------------------
def pack_features_to_regs(features):
    if len(features) != 70:
        raise ValueError(f"Expected 70 features, got {len(features)}")

    byte_vals = [(int(x) + 256) % 256 for x in features]

    regs = []
    for i in range(0, 70, 4):
        chunk = byte_vals[i:i+4]
        while len(chunk) < 4:
            chunk.append(0)
        word = (
            (chunk[0] << 0)  |
            (chunk[1] << 8)  |
            (chunk[2] << 16) |
            (chunk[3] << 24)
        )
        regs.append(word)

    while len(regs) < 18:
        regs.append(0)

    return regs

def mlp_write_features(features):
    regs = pack_features_to_regs(features)
    for i, word in enumerate(regs):
        addr = 0x04 + 4 * i
        mlp.write(addr, word)

def mlp_start_hold():
    mlp.write(0x00, 0x1)

def mlp_stop():
    mlp.write(0x00, 0x0)

def mlp_status():
    return mlp.read(0x00)

def mlp_done():
    return (mlp_status() >> 8) & 0x1

def mlp_predict(features, timeout_s=1.0):
    mlp_write_features(features)
    mlp_start_hold()

    t0 = time.time()
    while mlp_done() == 0:
        if time.time() - t0 > timeout_s:
            mlp_stop()
            raise TimeoutError("Timed out waiting for accelerator")

    status = mlp_status()
    cid = (status >> 9) & 0x7
    label = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else f"unknown({cid})"

    mlp_stop()
    return cid, label, status

# -----------------------------
# Main loop
# -----------------------------
while True:
    data, addr = sock.recvfrom(65535)

    try:
        payload = json.loads(data.decode("utf-8"))
    except Exception as e:
        print("[error] invalid JSON:", e)
        continue

    if "features" not in payload:
        print("[error] no features field")
        continue

    features = payload["features"]
    reply_ip = payload.get("reply_ip", DEFAULT_REPLY_IP)
    reply_port = int(payload.get("reply_port", DEFAULT_REPLY_PORT))

    print(f"\n[udp] received {len(features)} features from {addr}")
    print("[udp] first 10 features:", features[:10])

    try:
        cid, label, status = mlp_predict(features)

        reply = {
            "class_id": cid,
            "label": label,
            "status": int(status)
        }

        sock.sendto(json.dumps(reply).encode("utf-8"), (reply_ip, reply_port))

        print("[fpga] predicted:", label)
        print("[fpga] class_id:", cid)
        print("[fpga] status:", hex(status))
        print(f"[udp] replied to {(reply_ip, reply_port)}")

    except Exception as e:
        err = {
            "error": str(e)
        }
        sock.sendto(json.dumps(err).encode("utf-8"), (reply_ip, reply_port))
        print("[error] inference failed:", e)
 

[udp] listening on 0.0.0.0:5005

[udp] received 70 features from ('192.168.2.1', 65413)
[udp] first 10 features: [22, -64, 21, -59, 20, -55, 19, -51, 18, -46]
[fpga] predicted: line
[fpga] class_id: 3
[fpga] status: 0x701
[udp] replied to ('192.168.2.1', 5006)

[udp] received 70 features from ('192.168.2.1', 65413)
[udp] first 10 features: [-58, -44, -60, -37, -61, -30, -63, -22, -63, -15]
[fpga] predicted: freehand
[fpga] class_id: 4
[fpga] status: 0x901
[udp] replied to ('192.168.2.1', 5006)

[udp] received 70 features from ('192.168.2.1', 65413)
[udp] first 10 features: [64, -27, 59, -27, 54, -28, 49, -28, 44, -28]
[fpga] predicted: line
[fpga] class_id: 3
[fpga] status: 0x701
[udp] replied to ('192.168.2.1', 5006)

[udp] received 70 features from ('192.168.2.1', 65413)
[udp] first 10 features: [-64, -52, -62, -47, -60, -41, -58, -36, -56, -31]
[fpga] predicted: line
[fpga] class_id: 3
[fpga] status: 0x701
[udp] replied to ('192.168.2.1', 5006)

[udp] received 70 features from ('192


KeyboardInterrupt



In [4]:
# ── MLP Benchmark (HW vs SW baseline) ────────────────────────────────────────
import time
import json
from pathlib import Path

CLASS_NAMES = ["circle", "rectangle", "triangle", "line", "freehand"]

# ── SW baseline: pure Python integer MLP (mirrors fixed_point_ref.py) ─────────
def relu(x):
    return x if x > 0 else 0

def quantize_input(vec, scale=64):
    q = []
    for x in vec:
        val = int(round(x * scale))
        val = max(-128, min(127, val))
        q.append(val)
    return q

def mlp_infer_sw(x, params):
    fc1_w = params["fc1_weight"]
    fc1_b = params["fc1_bias"]
    fc2_w = params["fc2_weight"]
    fc2_b = params["fc2_bias"]

    hidden = []
    for j in range(16):
        acc = fc1_b[j]
        for i in range(70):
            acc += x[i] * fc1_w[j][i]
        hidden.append(relu(acc))

    outputs = []
    for k in range(5):
        acc = fc2_b[k]
        for j in range(16):
            acc += hidden[j] * fc2_w[k][j]
        outputs.append(acc)

    class_id = max(range(5), key=lambda k: outputs[k])
    return class_id

# ── HW inference (uses mlp_predict already defined in your notebook) ──────────
def benchmark_mlp(features_float, params, runs=100):
    """
    features_float : the raw float vector from preprocess_to_vector (length 70)
    params         : the loaded mlp_quantized.json dict
    runs           : number of repetitions for timing
    """
    x_q = quantize_input(features_float, scale=64)

    # ── SW timing ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    for _ in range(runs):
        sw_class = mlp_infer_sw(x_q, params)
    sw_ms = (time.perf_counter() - t0) / runs * 1000

    # ── HW timing ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    for _ in range(runs):
        hw_class, hw_label, _ = mlp_predict(x_q)
    hw_ms = (time.perf_counter() - t0) / runs * 1000

    speedup = sw_ms / hw_ms

    # ── Agreement check ───────────────────────────────────────────────────────
    agree = (sw_class == hw_class)

    # ── Print results ─────────────────────────────────────────────────────────
    print(f"{'Stage':<22} {'HW (ms)':>10}  {'SW (ms)':>10}  {'Speedup':>8}")
    print("-" * 56)
    print(f"  {'MLP inference':<20} {hw_ms:>10.4f}  {sw_ms:>10.4f}  {speedup:>7.2f}x")
    print()
    print(f"  SW predicted : {CLASS_NAMES[sw_class]} (class {sw_class})")
    print(f"  HW predicted : {hw_label} (class {hw_class})")
    print(f"  Agreement    : {'YES ✓' if agree else 'NO ✗ — MISMATCH'}")

    return hw_ms, sw_ms, agree

# ── Load weights and a test sample ───────────────────────────────────────────
WEIGHTS_PATH = Path("/home/xilinx/jupyter_notebooks/mlp/mlp_quantized.json")
SAMPLE_PATH  = Path("/home/xilinx/jupyter_notebooks/mlp/circle_1773088984993.json")

with open(WEIGHTS_PATH) as f:
    params = json.load(f)

with open(SAMPLE_PATH) as f:
    sample = json.load(f)

from preprocess import preprocess_to_vector

features = preprocess_to_vector(sample["stroke"], num_points=32, min_distance=2.0)

benchmark_mlp(features, params, runs=100)

Stage                     HW (ms)     SW (ms)   Speedup
--------------------------------------------------------
  MLP inference            0.9749      2.3221     2.38x

  SW predicted : circle (class 0)
  HW predicted : circle (class 0)
  Agreement    : YES ✓


(0.9748902100000123, 2.3221344600005978, True)

In [4]:
1+1

2